# PetMind — YOLOv8 행동 인식 학습 (Kaggle)

### 실행 전 필수 설정
1. 오른쪽 사이드바 → **Session options** → **Internet** 토글 켜기
2. 오른쪽 사이드바 → **Accelerator** → **GPU T4 x2** 선택
3. 왼쪽 사이드바 → **Secrets** → **Add a new secret**
   - Name: `ROBOFLOW_API_KEY`
   - Value: (본인 Roboflow API 키 입력)
4. 위에서 아래로 셀 순서대로 실행

In [ ]:
# 1. GPU 확인
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# 2. 패키지 설치
!pip install -q ultralytics roboflow

In [ ]:
# 3. Roboflow API 키 로드 (Kaggle Secrets)
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
ROBOFLOW_API_KEY = secrets.get_secret('ROBOFLOW_API_KEY')
print('API 키 로드 완료')

In [ ]:
# 4. 포즈 데이터셋 다운로드 (playing, resting, alert)
from roboflow import Roboflow
import os

os.chdir('/kaggle/working')

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('dog-pose-annotation').project('dog-pose-feaal')
version = project.version(12)
dataset = version.download('yolov8', location='/kaggle/working/pose_data')
print('포즈 데이터셋 다운로드 완료')

In [ ]:
# 5. 감정 데이터셋 다운로드 (happy, anxious)
project2 = rf.workspace('dog-emotion-zaveh').project('dog-emotion-ovhny')
version2 = project2.version(2)
dataset2 = version2.download('yolov8', location='/kaggle/working/emotion_data')
print('감정 데이터셋 다운로드 완료')

In [ ]:
# 6. 데이터 병합 및 라벨 매핑
import shutil
import yaml
from pathlib import Path

# 행동 클래스: 0=happy, 1=anxious, 2=playing, 3=resting, 4=alert
POSE_LABEL_MAP = {
    0: 2,  # playing
    1: 3,  # resting
    2: 4,  # alert
}
HAPPY_IDX = 0
ANXIOUS_IDX = 1

DATA_DIR = Path('/kaggle/working/behavior_data')
for split in ['train', 'val', 'test']:
    (DATA_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (DATA_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

def remap_and_copy(src_img_dir, src_lbl_dir, dst_split, label_map=None,
                   keep_classes=None, remap_to=None):
    src_img_dir = Path(src_img_dir)
    src_lbl_dir = Path(src_lbl_dir)
    if not src_img_dir.exists():
        return
    count = 0
    for img_path in src_img_dir.glob('*.*'):
        lbl_path = src_lbl_dir / (img_path.stem + '.txt')
        if not lbl_path.exists():
            continue
        lines = lbl_path.read_text().strip().splitlines()
        new_lines = []
        for line in lines:
            parts = line.split()
            if not parts:
                continue
            cls = int(parts[0])
            if label_map and cls in label_map:
                parts[0] = str(label_map[cls])
                new_lines.append(' '.join(parts))
            elif keep_classes and cls in keep_classes:
                parts[0] = str(remap_to[keep_classes.index(cls)])
                new_lines.append(' '.join(parts))
        if new_lines:
            dst_img = DATA_DIR / 'images' / dst_split / img_path.name
            dst_lbl = DATA_DIR / 'labels' / dst_split / (img_path.stem + '.txt')
            shutil.copy2(img_path, dst_img)
            dst_lbl.write_text('\n'.join(new_lines))
            count += 1
    print(f'  {dst_split}: {count}개 복사')

# 포즈 데이터 병합
print('포즈 데이터 병합 중...')
pose_base = Path('/kaggle/working/pose_data')
for src_split, dst_split in [('train','train'), ('valid','val'), ('test','test')]:
    remap_and_copy(
        pose_base / src_split / 'images',
        pose_base / src_split / 'labels',
        dst_split,
        label_map=POSE_LABEL_MAP
    )

# 감정 데이터에서 happy, sad 클래스 병합
print('감정 데이터 병합 중...')
emotion_base = Path('/kaggle/working/emotion_data')
with open(emotion_base / 'data.yaml') as f:
    emotion_yaml = yaml.safe_load(f)
emotion_names = emotion_yaml.get('names', [])
print('  감정 클래스:', emotion_names)

try:
    happy_src = emotion_names.index('happy')
    sad_src = emotion_names.index('sad')
except ValueError:
    happy_src, sad_src = 0, 1

for src_split, dst_split in [('train','train'), ('valid','val'), ('test','test')]:
    remap_and_copy(
        emotion_base / src_split / 'images',
        emotion_base / src_split / 'labels',
        dst_split,
        keep_classes=[happy_src, sad_src],
        remap_to=[HAPPY_IDX, ANXIOUS_IDX]
    )

print('데이터 병합 완료!')

In [ ]:
# 7. 클래스 분포 확인
from collections import Counter
from pathlib import Path

LABELS = ['happy', 'anxious', 'playing', 'resting', 'alert']
DATA_DIR = Path('/kaggle/working/behavior_data')

for split in ['train', 'val', 'test']:
    lbl_dir = DATA_DIR / 'labels' / split
    counts = Counter()
    for lbl_file in lbl_dir.glob('*.txt'):
        for line in lbl_file.read_text().splitlines():
            if line.strip():
                counts[int(line.split()[0])] += 1
    total = sum(counts.values())
    print(f'\n[{split}] 총 {total}개')
    for i, name in enumerate(LABELS):
        bar = '█' * (counts[i] // 100)
        print(f'  {name:12s} {counts[i]:5d}  {bar}')

In [ ]:
# 8. dataset.yaml 생성
import yaml

dataset_cfg = {
    'path': '/kaggle/working/behavior_data',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 5,
    'names': {0: 'happy', 1: 'anxious', 2: 'playing', 3: 'resting', 4: 'alert'}
}

with open('/kaggle/working/dataset.yaml', 'w') as f:
    yaml.dump(dataset_cfg, f, allow_unicode=True)

print('dataset.yaml 생성 완료')
!cat /kaggle/working/dataset.yaml

In [ ]:
# 9. YOLOv8 학습 시작
# ⏱️ 예상 시간: T4 기준 약 1.5~2시간 (100 에폭)
# 이 셀이 실행 중이면 컴퓨터를 꺼도 됩니다.
from ultralytics import YOLO
import torch

device = '0' if torch.cuda.is_available() else 'cpu'
print(f'학습 장치: {device}')

model = YOLO('yolov8n.pt')
results = model.train(
    data='/kaggle/working/dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=32,
    device=device,
    project='/kaggle/working/weights',
    name='behavior_v1',
    patience=20,
    save=True,
    val=True,
    fl_gamma=1.5,
    cos_lr=True,
    seed=42,
    plots=True,
)

print('\n학습 완료!')
print(f'mAP50: {results.results_dict["metrics/mAP50(B)"]:.4f}')

In [ ]:
# 10. 결과 확인 및 가중치 위치 안내
import os
from pathlib import Path

weight_dir = Path('/kaggle/working/weights/behavior_v1/weights')
best_pt = weight_dir / 'best.pt'

if best_pt.exists():
    size_mb = best_pt.stat().st_size / 1024 / 1024
    print(f'best.pt: {size_mb:.1f} MB')
    print(f'경로: {best_pt}')
    print('\n오른쪽 사이드바 → Output 탭 → weights 폴더에서 다운로드 가능')
else:
    print('best.pt 없음 — 학습 중 오류 확인 필요')

# 학습 결과 그래프 출력
from IPython.display import Image, display
results_png = Path('/kaggle/working/weights/behavior_v1/results.png')
if results_png.exists():
    display(Image(str(results_png)))